# 1장 1강: 문제 유형·데이터 분리·평가지표 리마인드

## 실습 목표

- Ames Housing의 `SalePrice`를 예측하는 문제가 어떤 머신러닝 문제 유형인지 구분합니다.
- 입력 데이터 `X`와 정답 `y`를 구분합니다.
- 데이터를 `train / valid / test`로 분리하고 각 데이터의 역할을 확인합니다.
- 회귀 평가지표 `MSE`, `RMSE`, `MAE`, `R²`의 의미를 확인합니다.
- 교안에서 소개한 **평균 예측 baseline**을 기준으로 기본 성능을 해석합니다.

> **실습 범위 안내**  
> 이번 실습은 1장 1강 교안의 리마인드 범위만 사용합니다.  
> 별도의 회귀 알고리즘, 교차검증, 하이퍼파라미터 튜닝, 새로운 평가지표는 사용하지 않습니다.

## 실습 환경 / 데이터

- Python
- pandas, numpy, scikit-learn
- 데이터: `Ames_Housing.csv`
- 예측 대상: `SalePrice` (주택 판매 가격)
- 이번 실습에서 확인할 입력 컬럼: `GrLivArea`, `OverallQual`, `YearBuilt`


## 실습 준비

먼저 데이터를 불러온 뒤 이번 실습에서 사용할 컬럼을 확인합니다.

- `GrLivArea`: 지상 생활 면적
- `OverallQual`: 주택의 전반적인 품질
- `YearBuilt`: 건축 연도
- `SalePrice`: 주택 판매 가격

이번 강의의 핵심은 **문제 유형, 데이터 분리, 평가지표**이므로 복잡한 전처리는 진행하지 않습니다.


In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

df = pd.read_csv("Ames_Housing.csv")

use_cols = ["GrLivArea", "OverallQual", "YearBuilt", "SalePrice"]
data = df[use_cols].copy()

print("데이터 크기:", data.shape)
display(data.head())


데이터 크기: (1460, 4)


,GrLivArea,OverallQual,YearBuilt,SalePrice
0,1710,7,2003,208500
1,1262,6,1976,181500
2,1786,7,2001,223500
3,1717,7,1915,140000
4,2198,8,2000,250000


In [2]:
print("[자료형]")
display(data.dtypes.to_frame("dtype"))

print("[결측치 수]")
display(data.isna().sum().to_frame("missing_count"))

print("[기초 통계량]")
display(data.describe())


[자료형]


,dtype
GrLivArea,int64
OverallQual,int64
YearBuilt,int64
SalePrice,int64


[결측치 수]


,missing_count
GrLivArea,0
OverallQual,0
YearBuilt,0
SalePrice,0


[기초 통계량]


,GrLivArea,OverallQual,YearBuilt,SalePrice
count,1460.000000,1460.000000,1460.000000,1460.000000
mean,1515.463699,6.099315,1971.267808,180921.195890
std,525.480383,1.382997,30.202904,79442.502883
min,334.000000,1.000000,1872.000000,34900.000000
25%,1129.500000,5.000000,1954.000000,129975.000000
50%,1464.000000,6.000000,1973.000000,163000.000000
75%,1776.750000,7.000000,2000.000000,214000.000000
max,5642.000000,10.000000,2010.000000,755000.000000


---

# 필수 1. 문제 유형과 train / valid / test 분리

## 배경

Ames Housing 데이터에서 주택의 특성을 이용해 `SalePrice`를 예측하려고 합니다.

교안에서 머신러닝 문제는 회귀·분류·군집화·차원축소로 구분했고, 지도학습에서는 입력 `X`와 정답 `y`를 사용합니다. 또한 모델의 일반화 성능을 확인하기 위해 데이터를 `train / valid / test`로 나눕니다.

## 문제

### 요구사항

1. `SalePrice` 예측이 회귀·분류·군집화·차원축소 중 어떤 문제인지 작성하세요.
2. 아래 기준으로 `X`와 `y`를 만드세요.
   - X: `GrLivArea`, `OverallQual`, `YearBuilt`
   - y: `SalePrice`
3. 교안의 예시와 같이 데이터를 약 **6 : 2 : 2** 비율의 `train / valid / test`로 나누세요.
   - 첫 번째 분리: `test_size=0.2`, `random_state=42`
   - 두 번째 분리: 남은 데이터에서 `test_size=0.25`, `random_state=42`
4. `X_train`, `X_valid`, `X_test`의 행 개수를 출력하세요.
5. 아래 질문에 답하세요.

### Q&A

- **Q1.** `SalePrice` 예측은 왜 해당 문제 유형에 속하나요?
- **Q2.** train 데이터는 어떤 목적으로 사용하나요?
- **Q3.** valid 데이터는 어떤 목적으로 사용하나요?
- **Q4.** test 데이터는 어떤 목적으로 사용하나요?

### 확인 포인트

- 정답 `y`가 존재하는 지도학습 문제인지 확인했는가?
- `SalePrice`가 연속적인 숫자 값이라는 점을 근거로 문제 유형을 판단했는가?
- train / valid / test의 역할을 구분했는가?


In [3]:
from sklearn.model_selection import train_test_split

# TODO 1. X와 y를 만드세요.
X = data[["GrLivArea", "OverallQual", "YearBuilt"]]
y = data["SalePrice"]

# TODO 2. train+valid / test로 먼저 분리하세요.
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# TODO 3. 남은 데이터를 train / valid로 분리하세요.
X_train, X_valid, y_train, y_valid = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42
)

# TODO 4. 각 데이터의 행 개수를 출력하세요.
print("X_train:", X_train.shape[0])
print("X_valid:", X_valid.shape[0])
print("X_test:", X_test.shape[0])

X_train: 876
X_valid: 292
X_test: 292


### 필수 1 답변 작성

- **Q1. 문제 유형과 이유:** SalePrice는 정해진 몇 개의 카테고리가 아니라 연속적인 숫자값이기 때문에 회귀 문제이다.
- **Q2. train의 역할:** 모델이 실제로 학습하는 데 사용하는 데이터 (876개).
- **Q3. valid의 역할:** 학습에는 쓰지 않고 중간에 성능을 확인하거나 여러 옵션을 비교하는 데 쓰는 데이터 (292개).
- **Q4. test의 역할:** 모든 결정이 끝난 뒤 마지막에 딱 한 번만 사용해 진짜 일반화 성능을 확인하는 데이터 (292개).


---

# 필수 2. 평균 예측 baseline과 회귀 평가지표

## 배경

교안에서는 회귀 문제를 `MSE`, `RMSE`, `MAE`, `R²`로 평가할 수 있다고 배웠습니다. 또한 R²를 설명하면서 **아무 특성도 사용하지 않고 y의 평균값만 예측하는 기준 모델(baseline)**을 소개했습니다.

이번 문제에서는 새로운 모델 알고리즘을 사용하지 않고, **train의 SalePrice 평균을 모든 valid 데이터의 예측값으로 사용하는 baseline**을 만들어 네 가지 회귀 평가지표를 확인합니다.

## 문제

### 요구사항

1. `y_train`의 평균을 `baseline_value`에 저장하세요.
2. `baseline_value`를 valid 데이터 개수만큼 반복하여 `y_valid_pred`를 만드세요.
3. `MSE`, `RMSE`, `MAE`, `R²`를 계산하세요.
4. 네 가지 결과를 출력하세요.
5. 아래 질문에 답하세요.

### Q&A

- **Q1.** MSE는 무엇을 의미하나요?
- **Q2.** RMSE는 무엇을 의미하며 MSE와 어떤 차이가 있나요?
- **Q3.** MAE는 무엇을 의미하나요?
- **Q4.** R²가 0에 매우 가깝다면 이 baseline을 어떻게 해석할 수 있나요?
- **Q5.** RMSE가 MAE보다 크게 나오는 이유를 교안의 설명을 이용해 작성하세요.

### 확인 포인트

- baseline 값은 반드시 `y_train`의 평균으로 만들었는가?
- MSE, RMSE, MAE, R²만 사용했는가?
- RMSE가 큰 오차에 더 큰 영향을 받는다는 점을 설명했는가?


In [4]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

# TODO 1. y_train 평균으로 baseline 값을 만드세요.
baseline_value = y_train.mean()

# TODO 2. valid 데이터 전체에 같은 baseline 값을 예측하도록 만드세요.
y_valid_pred = np.full(shape=y_valid.shape[0], fill_value=baseline_value)

# TODO 3. MSE, RMSE, MAE, R²를 계산하세요.
mse = mean_squared_error(y_valid, y_valid_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_valid, y_valid_pred)
r2 = r2_score(y_valid, y_valid_pred)

# TODO 4. 결과를 출력하세요.
print("MSE:", mse)
print("RMSE:", rmse)
print("MAE:", mae)
print("R2:", r2)

MSE: 5989110885.717651
RMSE: 77389.34607371774
MAE: 57793.59893976355
R2: -0.00011512063522900462


### 필수 2 답변 작성

- **Q1. MSE:** 예측값과 실제값 차이(오차)를 제곱해서 평균 낸 값. 여기선 약 59.9억으로, 단위가 원의 제곱이라 감이 잘 안 온다.
- **Q2. RMSE:** MSE에 루트를 씌운 값(77,389원). 원래 SalePrice와 같은 '원' 단위로 돌아와서, 평균적으로 약 7만 7천원 정도 오차가 난다고 직관적으로 해석할 수 있다.
- **Q3. MAE:** 오차의 절댓값을 평균 낸 값(57,794원). RMSE보다 작게 나오는데, 이는 큰 오차의 영향을 덜 받기 때문이다.
- **Q4. R² 해석:** 실제로 R2가 -0.000115로 거의 0이 나왔다. 이는 baseline이 SalePrice의 변동을 전혀 설명하지 못한다는 뜻이다. 특성을 하나도 안 쓴 것과 다름없으니, 나중에 만들 진짜 모델은 최소한 이 수준(R²≈0)보다는 확실히 좋아야 의미가 있다.
- **Q5. RMSE와 MAE 차이 해석:** 실제로 RMSE(77,389) > MAE(57,794)로 확인됐다. RMSE는 오차를 제곱해서 평균 내기 때문에 오차가 큰 값이 있으면 그 영향을 훨씬 크게 받는다. MAE는 절댓값이라 오차 크기만큼만 반영되므로, 두 값 사이에 약 2만원 차이가 나는 건 데이터에 오차가 큰 이상치들이 섞여있다는 신호다.


---

# 과제 1. 테스트 데이터에서 baseline 최종 확인

## 배경

교안에서는 test 데이터를 **최종 성능을 마지막에 한 번 확인하는 데이터**라고 설명했습니다.

필수 2에서 사용한 평균 예측 baseline을 그대로 유지한 상태에서 test 데이터의 성능을 확인해보세요. 새로운 모델이나 새로운 평가 방법은 사용하지 않습니다.

## 문제

### 요구사항

1. 필수 2에서 만든 `baseline_value`를 그대로 사용하세요.
2. test 데이터 개수만큼 동일한 값을 예측하여 `y_test_pred`를 만드세요.
3. test 데이터에 대해 `MSE`, `RMSE`, `MAE`, `R²`를 계산하세요.
4. 결과를 출력하세요.
5. 아래 질문에 답하세요.

### Q&A

- **Q1.** 이번 과제에서 test 데이터를 사용하는 목적은 무엇인가요?
- **Q2.** test의 R²가 0에 가깝거나 음수라면 평균 예측 baseline의 성능을 어떻게 해석할 수 있나요?
- **Q3.** RMSE와 MAE 중 큰 오차에 더 큰 벌점을 주는 지표는 무엇이며, 왜 그런가요?

### 확인 포인트

- test 데이터에서 새로운 값을 학습하거나 기준을 다시 정하지 않았는가?
- `y_train`에서 구한 baseline 값을 그대로 사용했는가?
- 교안의 회귀 평가지표 범위만 사용했는가?


In [5]:
# TODO 1. test 데이터의 baseline 예측값을 만드세요.
y_test_pred = np.full(shape=y_test.shape[0], fill_value=baseline_value)

# TODO 2. test 데이터의 MSE, RMSE, MAE, R²를 계산하세요.
mse_test = mean_squared_error(y_test, y_test_pred)
rmse_test = np.sqrt(mse_test)
mae_test = mean_absolute_error(y_test, y_test_pred)
r2_test = r2_score(y_test, y_test_pred)

# TODO 3. 결과를 출력하세요.
print("Test MSE:", mse_test)
print("Test RMSE:", rmse_test)
print("Test MAE:", mae_test)
print("Test R2:", r2_test)

Test MSE: 7676058192.019802
Test RMSE: 87613.11655237361
Test MAE: 62516.21340776882
Test R2: -0.0007472934889400928


### 과제 1 답변 작성

- **Q1. test 데이터의 목적:** 학습/검증 어디에도 쓰지 않은 완전히 새로운 데이터로 baseline의 진짜 일반화 성능을 마지막에 딱 한 번 확인하기 위해서다.
- **Q2. R² 해석:** test에서도 R2가 -0.000747로 여전히 거의 0(음수)이다. baseline이 test 데이터에서도 설명력이 전혀 없고, 오히려 아주 살짝 평균만 찍는 것보다도 못하다는 뜻이다. → valid에서 확인한 것과 결론이 같다.
- **Q3. 큰 오차에 더 큰 벌점을 주는 지표와 이유:** RMSE다. test에서도 RMSE(87,613) > MAE(62,516)로 약 2만 5천원 차이가 나는데, 오차를 제곱하는 과정에서 큰 오차(이상치)일수록 훨씬 크게 부풀려지기 때문이다.


---

# 실습 마무리

아래 세 가지를 한 문장씩 정리해보세요.

1. **어떤 분석 문제가 있었는가?** Ames Housing 데이터로 주택 특성(GrLivArea, OverallQual, YearBuilt)을 이용해 SalePrice(연속값)를 예측하는 회귀 문제였다. 
2. **어떤 데이터 분리와 평가지표를 적용했는가?** 데이터를 876(train) : 292(valid) : 292(test) = 6:2:2로 나누고, MSE·RMSE·MAE·R² 네 가지 회귀 평가지표로 성능을 확인했다.
3. **평균 예측 baseline의 결과를 무엇을 근거로 해석했는가?** valid와 test 모두 R²가 0에 매우 가깝거나 음수(-0.0001, -0.0007)로 나와 baseline이 설명력이 거의 없다는 걸 확인했고, RMSE가 MAE보다 항상 크게 나온 걸 근거로 데이터에 오차가 큰 이상치들이 존재한다는 것도 함께 해석했다.
